In [1]:
import pandas as pd

In [2]:
active = pd.read_csv("active_minutes.csv")
glucose = pd.read_csv("glucose.csv")
sleep = pd.read_csv("sleep.csv")
sleep_score = pd.read_csv("sleep_score.csv")
rhr = pd.read_csv("resting_heart_rate.csv")
hormones = pd.read_csv("hormones_and_selfreport.csv")

In [3]:
# getting data only for 2022
# put all datasets in one dictionary
datasets = {
    "active": active,
    "glucose": glucose,
    "sleep": sleep,
    "rhr": rhr,
    "sleep_score": sleep_score,
    "hormones": hormones
}

# keep only 2022 rows in each dataset
for name in datasets:
    datasets[name] = datasets[name][datasets[name]["study_interval"] == 2022]

# put them back into variables if you want
active = datasets["active"]
glucose = datasets["glucose"]
sleep = datasets["sleep"]
rhr = datasets["rhr"]
sleep_score = datasets["sleep_score"]
hormones = datasets["hormones"]

In [4]:
keys = ["id", "day_in_study"]

In [5]:
# cleaning active_minutes into 2 categories: sedentary vs active
# one "active" column first
active["active_mins"] = (
    active["lightly"].fillna(0)
    + active["moderately"].fillna(0)
    + active["very"].fillna(0)
)


active2 = active[["id", "day_in_study", "sedentary", "active_mins"]]
active_daily = active2.groupby(["id", "day_in_study"]).sum()
active_daily = active_daily.reset_index()
active_daily = active_daily.rename(columns={"sedentary": "sedentary_mins"})

In [6]:
# cleaning glucose for daily mean per id
glucose2 = glucose[["id", "day_in_study", "glucose_value"]]
glucose_daily = glucose2.groupby(["id", "day_in_study"]).mean()
glucose_daily = glucose_daily.reset_index()
glucose_daily = glucose_daily.rename(columns={"glucose_value": "glucose_mean"})

In [7]:
# cleaning resting heart rate for daily mean per id
rhr2 = rhr[["id", "day_in_study", "value"]]
rhr_daily = rhr2.groupby(["id", "day_in_study"]).mean()
rhr_daily = rhr_daily.reset_index()
rhr_daily = rhr_daily.rename(columns={"value": "rhr_mean"})

In [8]:
# daily sleep variables
# minutes asleep
asleep = sleep[["id", "sleep_start_day_in_study", "minutesasleep"]]
asleep_daily = asleep.groupby(["id", "sleep_start_day_in_study"]).sum()
asleep_daily = asleep_daily.reset_index()
asleep_daily = asleep_daily.rename(columns={"minutesasleep": "minutes_asleep"})

# minutes awake
awake = sleep[["id", "sleep_start_day_in_study", "minutesawake"]]
awake_daily = awake.groupby(["id", "sleep_start_day_in_study"]).sum()
awake_daily = awake_daily.reset_index()
awake_daily = awake_daily.rename(columns={"minutesawake": "minutes_awake"})

# sleep efficiency
sleep_eff = sleep[["id", "sleep_start_day_in_study", "efficiency"]]
sleep_eff_daily = sleep_eff.groupby(["id", "sleep_start_day_in_study"]).mean()
sleep_eff_daily = sleep_eff_daily.reset_index()
sleep_eff_daily = sleep_eff_daily.rename(columns={"efficiency": "sleep_efficiency_mean"})

# combine the sleep summaries
sleep_daily = asleep_daily.merge(awake_daily, on=["id", "sleep_start_day_in_study"], how="outer")
sleep_daily = sleep_daily.merge(sleep_eff_daily, on=["id", "sleep_start_day_in_study"], how="outer")
sleep_daily = sleep_daily.rename(columns={"sleep_start_day_in_study": "day_in_study"})

In [9]:
# sleep score; overall, revitalization, and duration scores
sleep_score_subset = sleep_score[["id", "day_in_study", "overall_score", "revitalization_score", "duration_score"]]
sleep_score_daily = sleep_score_subset.groupby(["id", "day_in_study"]).mean()
sleep_score_daily = sleep_score_daily.reset_index()
sleep_score_daily = sleep_score_daily.rename(columns={
    "overall_score": "sleep_score_overall",
    "revitalization_score": "sleep_score_revitalization",
    "duration_score": "sleep_score_duration"
})

In [10]:
# merge into one dataset
data = active_daily.merge(glucose_daily, on=["id", "day_in_study"], how="outer")
data = data.merge(rhr_daily, on=["id", "day_in_study"], how="outer")
data = data.merge(sleep_daily, on=["id", "day_in_study"], how="outer")
data = data.merge(sleep_score_daily, on=["id", "day_in_study"], how="outer")
data = data.merge(hormones, on=["id", "day_in_study"], how="outer")

In [ ]:
data

,id,day_in_study,sedentary_mins,active_mins,glucose_mean,rhr_mean,minutes_asleep,minutes_awake,sleep_efficiency_mean,sleep_score_overall,...,headaches,cramps,sorebreasts,fatigue,sleepissue,moodswing,stress,foodcravings,indigestion,bloating
0,1,1,753.0,64,5.498958,74.785346,1027.0,26.0,98.0,NaN,...,High,Very Low/Little,Very Low/Little,High,Low,Very Low/Little,Moderate,Very Low/Little,Very Low/Little,Very Low/Little
1,1,2,855.0,74,5.372222,80.407307,77.0,4.0,95.0,NaN,...,Very High,Very Low/Little,Very Low/Little,High,Very High,Very Low/Little,Moderate,Very Low/Little,Very Low/Little,Very Low/Little
2,1,3,751.0,159,5.579514,84.686869,502.0,28.0,95.0,NaN,...,High,Very Low/Little,Very Low/Little,Very High,Very High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
3,1,4,905.0,86,5.206597,83.852219,384.0,65.0,93.0,80.0,...,Very Low/Little,Very Low/Little,Very Low/Little,High,Very High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
4,1,5,1430.0,10,5.381597,0.000000,NaN,NaN,NaN,NaN,...,Very Low/Little,Very Low/Little,Very Low/Little,High,High,Very Low/Little,Low,Very Low/Little,Very Low/Little,Very Low/Little
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
3693,50,86,907.0,252,6.570486,58.132595,230.0,51.0,90.0,56.0,...,Low,Not at all,Not at all,Very High,Low,High,Moderate,High,Moderate,Low
3694,50,87,615.0,302,6.661806,57.626328,475.0,48.0,97.0,84.0,...,Very Low/Little,Not at all,Not at all,Moderate,Low,Low,Moderate,Low,Low,Low
3695,50,88,800.0,181,7.151389,56.868725,396.0,63.0,92.0,69.0,...,Very Low/Little,Very Low/Little,Moderate,Moderate,High,Moderate,Moderate,Very High,Moderate,Moderate
3696,50,89,760.0,267,7.118750,56.439247,362.0,51.0,95.0,74.0,...,Not at all,Low,Low,Moderate,Moderate,Moderate,Moderate,Very High,Moderate,Moderate


In [12]:
# checking that 2022 filter works
print(active["day_in_study"].max())
print(glucose["day_in_study"].max())
print(sleep["sleep_start_day_in_study"].max())
print(rhr["day_in_study"].max())
print(sleep_score["day_in_study"].max())
print(hormones["day_in_study"].max())

90
90
90
90
90
90


In [13]:
print("active rows:", active.shape[0])
print("glucose rows:", glucose.shape[0])
print("sleep rows:", sleep.shape[0])
print("rhr rows:", rhr.shape[0])
print("sleep_score rows:", sleep_score.shape[0])
print("hormones rows:", hormones.shape[0])

active rows: 3698
glucose rows: 837130
sleep rows: 4111
rhr rows: 3698
sleep_score rows: 3372
hormones rows: 3698


In [15]:
print("active rows:", active_daily.shape[0])
print("glucose rows:", glucose_daily.shape[0])
print("sleep rows:", sleep_daily.shape[0])
print("rhr rows:", rhr_daily.shape[0])
print("sleep_score rows:", sleep_score_daily.shape[0])
print("hormones rows:", hormones.shape[0])

active rows: 3698
glucose rows: 3109
sleep rows: 3172
rhr rows: 3698
sleep_score rows: 3248
hormones rows: 3698


In [ ]:
# save dataset
data.to_csv("data.csv", index=False)